# HDB resale — HistGradientBoosting baseline

Baseline regression with **`HistGradientBoostingRegressor`**, time-based validation, and primary metric **RMSE** (dollar scale).

**Submission outputs:**
- `ROOT / submission / sub_hgb_t.csv` (time split)
- `ROOT / submission / sub_hgb_r.csv` (random split)


In [ ]:
# Paths: submissions written to ROOT / submission / sub_hgb_{t,r}.csv
# CLI (from project root): conda run -n hdb-ml-env python run_hgb_baseline.py
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebook" else _cwd
TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUB_PATH = ROOT / "data" / "sample_sub_reg.csv"
SUBMISSION_PATH_T = ROOT / "submission" / "sub_hgb_t.csv"
SUBMISSION_PATH_R = ROOT / "submission" / "sub_hgb_r.csv"

print(f"ROOT: {ROOT.resolve()}")
print(f"Train: {TRAIN_PATH.resolve()}")
print(f"Test:  {TEST_PATH.resolve()}")
print(f"Time-split submission:   {SUBMISSION_PATH_T.resolve()}")
print(f"Random-split submission: {SUBMISSION_PATH_R.resolve()}")

train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)
print(train.shape, test.shape)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:  # pragma: no cover
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

RNG = 42
TARGET = "resale_price"

# Drop identifiers, target leakage helpers, and high-cardinality text (v1 baseline)
DROP_FEATURES = [
    "id",
    "block",  # high cardinality; HGBR categoricals must be <= 255 levels
    "full_flat_type",
    "street_name",
    "mrt_name",
    "bus_stop_name",
    "pri_sch_name",
    "sec_sch_name",
    "address",
]

feature_cols = [c for c in train.columns if c not in DROP_FEATURES and c != TARGET]
missing_in_test = set(feature_cols) - set(test.columns)
missing_in_train = set(test.columns) - set(train.columns) - {TARGET}
assert not missing_in_test, f"Features missing in test: {missing_in_test}"
print("Columns only in test (expected: none except target):", missing_in_train)

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y = train[TARGET].astype(float)

# Time column for split (not a model feature)
period = pd.to_datetime(train["Tranc_YearMonth"], format="%Y-%m")



In [ ]:
# Numeric vs categorical columns (post-drop)
num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]
# HGBR native categoricals must have cardinality <= 255; high-cardinality columns
# are ordinal-encoded but treated as numeric by the booster.
cat_low = [c for c in cat_cols if X_train[c].nunique(dropna=True) <= 255]
cat_high = [c for c in cat_cols if c not in cat_low]
print(
    f"Numeric: {len(num_cols)}, Categorical (<=255 levels): {len(cat_low)}, "
    f"Ordinal-as-numeric: {len(cat_high)}"
)
if cat_high:
    print("  High-cardinality object columns:", cat_high)

num_pipe = SimpleImputer(strategy="median")
cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        ),
    ]
)

transformers = [("num", num_pipe, num_cols)]
if cat_low:
    transformers.append(("cat_low", cat_pipe, cat_low))
if cat_high:
    transformers.append(("cat_high", cat_pipe, cat_high))

preprocess = ColumnTransformer(
    transformers=transformers,
    verbose_feature_names_out=False,
)

cat_idx = list(range(len(num_cols), len(num_cols) + len(cat_low)))
hgb_kw = dict(
    random_state=RNG,
    max_iter=300,
    learning_rate=0.08,
    max_depth=12,
    min_samples_leaf=20,
    l2_regularization=0.1,
)
if cat_idx:
    hgb_kw["categorical_features"] = cat_idx
hgb = HistGradientBoostingRegressor(**hgb_kw)

model = Pipeline(steps=[("prep", preprocess), ("hgb", hgb)])



In [ ]:
# Time-based holdout: last 12 months of training calendar
cutoff = period.max() - pd.DateOffset(months=12)
val_mask = period >= cutoff
print(f"Validation window: >= {cutoff.date()} (last 12 months in data)")
print(f"Train rows: {(~val_mask).sum():,}, Val rows: {val_mask.sum():,}")
model.fit(X_train.loc[~val_mask], y.loc[~val_mask])
y_val_pred = model.predict(X_train.loc[val_mask])
rmse = root_mean_squared_error(y[val_mask], y_val_pred)
mae = mean_absolute_error(y[val_mask], y_val_pred)
print(f"Validation RMSE (primary): {rmse:,.2f}")
print(f"Validation MAE (diagnostic): {mae:,.2f}")
from sklearn.base import clone
from sklearn.model_selection import train_test_split
X_tr_r, X_va_r, y_tr_r, y_va_r = train_test_split(
    X_train,
    y,
    test_size=0.30,
    random_state=RNG,
    shuffle=True,
)
model_rand_eval = clone(model)
model_rand_eval.fit(X_tr_r, y_tr_r)
y_va_r_pred = model_rand_eval.predict(X_va_r)
rmse_r = root_mean_squared_error(y_va_r, y_va_r_pred)
mae_r = mean_absolute_error(y_va_r, y_va_r_pred)
print(f"Random 70/30 holdout RMSE: {rmse_r:,.2f}")
print(f"Random 70/30 holdout MAE: {mae_r:,.2f}")


In [ ]:
# Train split-specific models; predict test; write Kaggle submissions (_t and _r)
from sklearn.base import clone
from sklearn.model_selection import train_test_split

# Time split model (trained on rows before last-12-month holdout)
model_time = clone(model)
model_time.fit(X_train.loc[~val_mask], y.loc[~val_mask])
test_pred_t = model_time.predict(X_test)

# Random split model (trained on 70% random fold)
X_tr_r, _, y_tr_r, _ = train_test_split(
    X_train,
    y,
    test_size=0.30,
    random_state=RNG,
    shuffle=True,
)
model_rand = clone(model)
model_rand.fit(X_tr_r, y_tr_r)
test_pred_r = model_rand.predict(X_test)

sample = pd.read_csv(SAMPLE_SUB_PATH, nrows=5)
sub_t = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_t})
sub_r = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_r})
assert list(sub_t.columns) == list(sample.columns), (sub_t.columns.tolist(), sample.columns.tolist())
assert list(sub_r.columns) == list(sample.columns), (sub_r.columns.tolist(), sample.columns.tolist())

SUBMISSION_PATH_T.parent.mkdir(parents=True, exist_ok=True)
sub_t.to_csv(SUBMISSION_PATH_T, index=False)
sub_r.to_csv(SUBMISSION_PATH_R, index=False)
print(f"Wrote {SUBMISSION_PATH_T.resolve()} ({len(sub_t):,} rows)")
print(f"Wrote {SUBMISSION_PATH_R.resolve()} ({len(sub_r):,} rows)")
print(sub_t.head())


## Next steps

- Add **target encoding** or grouping for `street_name` / block clusters instead of dropping text.
- Tune **HGBR** (`max_depth`, `learning_rate`, `max_iter`) with time-based CV.
- Try **log1p(target)** training; report RMSE in dollar space after `expm1` on predictions.
- Blend with **LightGBM** / **XGBoost**; add **SHAP** on the validation fold.

